# EEG for LLMs: A Telemetry Layer for Online Uncertainty Monitoring and Decision Policies

**Author:** Nikolay Yudin  
**Email:** n.yudin@gmail.com  
**Date:** 2026-01-30  
**Code:** https://github.com/nick-yudin/grokking-research  

This notebook runs the Paper 1 cascade demo (telemetry-gated accept / retry / route).

## Abstract
Token streams are a human-oriented interface that can obscure generation dynamics and encourage brittle analyses (e.g., relying on chain-of-thought text). We introduce an "EEG-like" telemetry layer for autoregressive decoding that records lightweight internal signals during generation—uncertainty, surprisal, distribution shift, and sparse layer summaries—yielding real-time traces of model state evolution without parsing chain-of-thought text. Across three model families and three task types (27 runs = 3 models × 3 tasks × 3 seeds), we find that telemetry signatures vary strongly across models and tasks, and that early-window uncertainty can predict failures above random on labeled tasks. As an application demo, we show how telemetry can gate a simple cascade policy (accept / retry / route) on a Llama-8B → Qwen-14B pair.


In [ ]:
# Install runtime dependencies
# Note: this notebook assumes internet access for pip and model downloads.

!pip -q install -U "transformers>=4.46" "datasets>=2.20" "accelerate" "matplotlib" "pandas" "tqdm" "seaborn"

import os
from pathlib import Path

REPO_DIR = Path('/content/grokking-research')
assert REPO_DIR.exists(), f"Repo not found at {REPO_DIR}. Upload and extract the repo to /content/grokking-research."

os.chdir(str(REPO_DIR))
print('Repo:', REPO_DIR)


In [ ]:
import subprocess
from pathlib import Path

# Where we store outputs (no Drive; local to this Colab runtime).
ARTIFACTS_ROOT = str(Path('/content') / 'artifacts')
Path(ARTIFACTS_ROOT).mkdir(parents=True, exist_ok=True)

def run(cmd: str):
    print('$', cmd)
    subprocess.run(cmd, shell=True, check=True)

# Experiment config (edit as needed)
BASE_MODEL = "meta-llama/Meta-Llama-3.1-8B-Instruct"
EXPERT_MODEL = "Qwen/Qwen2.5-14B-Instruct"
SEED = 0
N = 50  # Demo size. Increase to 200 for canonical.
RUN_ID = f"paper1_cascade_demo.llama8b_to_qwen14b.gsm8k.seed{SEED}.n{N}"

print('Artifacts root:', ARTIFACTS_ROOT)
print('RUN_ID:', RUN_ID)


In [ ]:
# Run GSM8K cascade (accept / retry / route) using an early entropy risk score.
run(
    " ".join(
        [
            "python3 -u latent_configuration/01_interpretable_latent_probes/telemetry_collect.py",
            f"--run_id {RUN_ID}",
            f"--artifacts_root {ARTIFACTS_ROOT}",
            f"--model_name \"{BASE_MODEL}\" --dtype bf16",
            "--dataset gsm8k --split test",
            f"--num_examples {N} --seed {SEED}",
            "--gsm8k_prompt_mode work_then_final",
            "--max_new_tokens 96",
            "--strict_final_format --format_mode strict --stop_after_final_number",
            "--topk 50 --layers auto:last4 --temperature 0.0",
            "--sample_log_every 0 --heartbeat_s 900",
            "--policy cascade",
            "--risk_feature mean_entropy_norm --risk_window 8",
            "--retry_threshold 0.44 --route_threshold 0.67",
            "--retry_pool v1 --retry_chooser majority_all_else_greedy_on_tie",
            f"--route_model_name \"{EXPERT_MODEL}\" --route_dtype bf16",
            "--route_max_new_tokens 96 --route_temperature 0.0",
        ]
    )
)


In [ ]:
run(
    " ".join(
        [
            "python3 -u latent_configuration/01_interpretable_latent_probes/telemetry_analyze.py",
            f"--run_dir {ARTIFACTS_ROOT}/runs/{RUN_ID}",
        ]
    )
)


In [ ]:
# Optional: visualize a few traces.
run(
    " ".join(
        [
            "python3 -u latent_configuration/01_interpretable_latent_probes/telemetry_viz.py",
            f"--run_dir {ARTIFACTS_ROOT}/runs/{RUN_ID}",
            "--max_plots 12 --plot_selector stratified --plots_per_bucket 3",
        ]
    )
)


In [ ]:
from pathlib import Path

report_path = Path(ARTIFACTS_ROOT) / 'runs' / RUN_ID / 'analysis' / 'report.md'
print('report:', report_path)
print(report_path.read_text(encoding='utf-8')[:5000])

viz_dir = Path(ARTIFACTS_ROOT) / 'runs' / RUN_ID / 'viz'
pngs = sorted(viz_dir.glob('*.png')) if viz_dir.exists() else []
print('viz pngs:', len(pngs))
if pngs:
    from IPython.display import Image, display
    display(Image(filename=str(pngs[0])))


## Download artifacts

Use the file browser to download:
- `/content/artifacts/runs/<RUN_ID>/analysis/` (reports)
- `/content/artifacts/runs/<RUN_ID>/viz/` (figures, if generated)
- `/content/artifacts/runs/<RUN_ID>/route_debug.jsonl` (expert outputs, if routed)
- `/content/artifacts/runs/<RUN_ID>/samples.jsonl` (per-sample records)
